## Difference-in-Differences Estimation

This notebook implements baseline Difference-in-Differences and event-study
specifications using the monthly HS8-level panel constructed in earlier steps.

### Estimation workflow (pseudocode)

- Load `did_panel_monthly.csv`
- Define panel unit: `product = hs8`
- Filter sample:
  - non-missing outcome
  - non-missing `post`, `highgap`, and `date`

For each outcome in:
- `ln_customs_value`
- `ln_first_unit_qty`
- `ln_unit_value`

Estimate the following models:

- **m1:**  
  `outcome ~ highgap × post`

- **m2:**  
  `outcome ~ highgap × post | product`

- **m3:**  
  `outcome ~ highgap × post | date`

- **m4:**  
  `outcome ~ highgap × post | product + date`

- Cluster standard errors by `product`
- Save results in a ladder (incremental FE) table

### Event-study specification

- Estimate:

      outcome ~ i(event_time, highgap, ref = -1) | product + date

- Cluster standard errors by `product`
- Save event-study plot with x-axis labeled in calendar years


In [97]:
import pandas as pd

df = pd.read_csv("../transformed_data/did_panel_monthly.csv")
df.shape

(824352, 17)

In [98]:
df_clean = df.dropna(
    subset=["ln_customs_value", "ln_first_unit_qty", "ln_unit_value", "post", "highgap", "date"]
)

df_clean.shape

(414730, 17)

In [99]:
# from pyfixest.estimation import feols
# from pyfixest.report import etable

# # m1: no fixed effects
# m1 = feols(
#     "ln_customs_value ~ highgap * post",
#     data=df_clean,
#     vcov={"CRV1": "hs8"}
# )

# # m2: product fixed effects
# m2 = feols(
#     "ln_customs_value ~ highgap * post | hs8",
#     data=df_clean,
#     vcov={"CRV1": "hs8"}
# )

# # m3: date fixed effects
# m3 = feols(
#     "ln_customs_value ~ highgap * post | date",
#     data=df_clean,
#     vcov={"CRV1": "hs8"}
# )

# # m4: product + date fixed effects
# m4 = feols(
#     "ln_customs_value ~ highgap * post | hs8 + date",
#     data=df_clean,
#     vcov={"CRV1": "hs8"}
# )


In [100]:
import pandas as pd
from linearmodels.panel import PanelOLS, compare

# 1. Convert 'date' to datetime objects so linearmodels is happy
df_clean['date'] = pd.to_datetime(df_clean['date'])

# 2. Set the MultiIndex [Entity, Time]
df_panel = df_clean.set_index(['hs8', 'date'])

# 3. Define and fit the models
# Note: Added 'cov_type="clustered", cluster_entity=True' to match your vcov={"CRV1": "hs8"}
# Model 1: No FE - Standard OLS
m1 = PanelOLS.from_formula("ln_customs_value ~ 1 + highgap * post", 
                           data=df_panel).fit(cov_type="clustered", cluster_entity=True)

# Model 2: Product FE - highgap is absorbed (it's constant per hs8)
m2 = PanelOLS.from_formula("ln_customs_value ~ post + highgap:post + EntityEffects", 
                           data=df_panel, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

# Model 3: Date FE - post is absorbed (it's constant per date)
m3 = PanelOLS.from_formula("ln_customs_value ~ highgap + highgap:post + TimeEffects", 
                           data=df_panel, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

# Model 4: Both FE - highgap and post are both absorbed
m4 = PanelOLS.from_formula("ln_customs_value ~ highgap:post + EntityEffects + TimeEffects", 
                           data=df_panel, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)

# Generate and save the TXT table
# Use 'stat' instead of 'statistics'
res_table = compare(
    {"m1_customs": m1, "m2_customs": m2, "m3_customs": m3, "m4_customs": m4}, 
    stars=True,
    precision='std_errors'  # This puts Standard Errors in the parentheses
)

# Convert to text and swap the labels to match your original preference
table_text = res_table.summary.as_text()
table_text = table_text.replace("Entity", "product").replace("Time", "date")

# Save to your file
with open("../results/did_ladder_customs.txt", "w") as f:
    f.write(table_text)

print("Table successfully exported with Standard Errors!")

/var/folders/hq/hfb1chlj2pxg9shnwqj9t7jw0000gn/T/ipykernel_14212/2842675765.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['date'] = pd.to_datetime(df_clean['date'])


Table successfully exported with Standard Errors!
